# Laboratoire 4 : Variational Quantum Classifier (VQC)

**Quantum Machine Learning — Master/PhD**

Semaine 5 — Circuits quantiques variationnels pour la classification

## Objectifs

- Implémenter un classifieur quantique variationnel complet sur le dataset Iris
- Maîtriser la structure : AngleEmbedding → BasicEntanglerLayers → mesure
- Entraîner avec la cross-entropy et l'optimiseur Adam
- Comparer les performances avec la baseline classique du Lab 2
- Explorer différents ansätze et stratégies de régularisation

## Bibliothèques requises

- `pennylane` (≥ 0.45)
- `qiskit` (≥ 2.0) et `qiskit-machine-learning`
- `torch` (≥ 2.x)
- `scikit-learn`
- `matplotlib`
- `numpy`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.svm import SVC

import pennylane as qml
from pennylane import numpy as pnp

import torch
import torch.nn as nn
import torch.optim as optim

print('Bibliothèques chargées avec succès.')
print(f'PennyLane : {qml.__version__}')
print(f'PyTorch   : {torch.__version__}')

---
## Partie A : Circuit VQC de base

Architecture :

1. **AngleEmbedding** : encode les 4 features d'Iris en rotations sur 4 qubits
2. **BasicEntanglerLayers** : couches variationnelles de rotations + portes CNOT
3. **Mesure** : espérance de $\sigma_Z$ sur un qubit pour la classification binaire

In [ ]:
# Configuration du VQC

n_qubits = 4
n_layers = 3

dev = qml.device('default.qubit', wires=n_qubits)

@qml.qnode(dev)
def vqc_circuit(inputs, weights):
    """
    Circuit VQC : AngleEmbedding + BasicEntanglerLayers + mesure.
    
    Args:
        inputs: données d'entrée (4 features)
        weights: paramètres variationnels
    """
    # Encodage
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation='Y')
    
    # Couches variationnelles
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    
    # Mesure : espérance de Pauli Z sur le premier qubit
    return qml.expval(qml.PauliZ(0))

# Test du circuit
np.random.seed(42)
test_inputs = pnp.array([0.5, 1.2, 0.8, 2.0], requires_grad=False)
init_weights = pnp.random.uniform(-np.pi, np.pi, (n_layers, n_qubits, 3), requires_grad=True)

output = vqc_circuit(test_inputs, init_weights)
print(f"Circuit VQC testé avec succès.")
print(f"Sortie (expectation Z) : {output:.4f}")
print(f"Forme des poids : {init_weights.shape}")
print()
print("Circuit VQC :")
print(qml.draw(vqc_circuit)(test_inputs, init_weights))

In [ ]:
# Préparation du dataset Iris (classification binaire)

iris = datasets.load_iris()
X, y = iris.data, iris.target

# On garde 2 classes pour la classification binaire
mask = y != 2
X_bin, y_bin = X[mask], y[mask]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_bin, y_bin, test_size=0.3, random_state=42, stratify=y_bin
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Conversion en PennyLane tensors
X_train_pl = pnp.array(X_train_scaled, requires_grad=False)
y_train_pl = pnp.array(y_train * 2 - 1, requires_grad=False)  # {0,1} -> {-1, +1}
X_test_pl = pnp.array(X_test_scaled, requires_grad=False)
y_test_orig = y_test

print(f"Dataset Iris (2 classes) :")
print(f"  Train : {X_train_pl.shape[0]} échantillons")
print(f"  Test  : {X_test_pl.shape[0]} échantillons")
print(f"  Labels : -1 (setosa), +1 (versicolor)")

---
## Partie B : Fonction de coût et optimisation

On utilise la **cross-entropy** comme fonction de coût :

$$\mathcal{L}(\theta) = -\frac{1}{N}\sum_{i=1}^{N} y_i \log(\hat{y}_i) + (1-y_i) \log(1-\hat{y}_i)$$

où $\hat{y}_i = \frac{1 + \langle Z \rangle_i}{2}$ est la probabilité prédite.

In [ ]:
# Fonction de coût

def sigmoid(x):
    return 1 / (1 + pnp.exp(-x))

def cross_entropy_loss(weights, X, y):
    """Cross-entropy pour la classification binaire."""
    predictions = []
    for i in range(len(X)):
        expval = vqc_circuit(X[i], weights)
        prob = sigmoid(expval)  # probabilité d'être en classe +1
        predictions.append(prob)
    predictions = pnp.array(predictions)
    
    # y est en {-1, +1}, on convertit en {0, 1}
    y_binary = (y + 1) / 2
    
    loss = -pnp.mean(y_binary * pnp.log(predictions + 1e-10) + 
                     (1 - y_binary) * pnp.log(1 - predictions + 1e-10))
    return loss

def accuracy_vqc(weights, X, y):
    """Accuracy du VQC."""
    correct = 0
    for i in range(len(X)):
        expval = vqc_circuit(X[i], weights)
        pred = 1 if expval >= 0 else -1
        if pred == y[i]:
            correct += 1
    return correct / len(X)

# Test initial
loss_init = cross_entropy_loss(init_weights, X_train_pl, y_train_pl)
acc_init = accuracy_vqc(init_weights, X_train_pl, y_train_pl)
print(f"Loss initiale : {loss_init:.4f}")
print(f"Accuracy initiale : {acc_init:.4f}")

In [ ]:
# Optimisation avec Adam (PennyLane)

opt = qml.AdamOptimizer(stepsize=0.1)
weights = init_weights

n_epochs = 100
loss_history = []
acc_history = []

print("Entraînement du VQC en cours...")
for epoch in range(n_epochs):
    weights, loss = opt.step_and_cost(
        lambda w: cross_entropy_loss(w, X_train_pl, y_train_pl),
        weights
    )
    loss_history.append(loss)
    
    if (epoch + 1) % 10 == 0:
        acc = accuracy_vqc(weights, X_train_pl, y_train_pl)
        acc_history.append(acc)
        print(f"Époque {epoch+1:3d}/{n_epochs} — Loss : {loss:.4f} — Accuracy : {acc:.4f}")

print("\nEntraînement terminé.")

In [ ]:
# Visualisation de l'entraînement

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(loss_history, label='Loss')
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('Cross-entropy')
axes[0].set_title('Courbe d\'apprentissage')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(np.arange(10, n_epochs + 1, 10), acc_history, 'o-', label='Accuracy')
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('Accuracy (train)')
axes[1].set_title('Accuracy pendant l\'entraînement')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Évaluation sur le test set

test_preds = []
for i in range(len(X_test_pl)):
    expval = vqc_circuit(X_test_pl[i], weights)
    pred = 1 if expval >= 0 else -1
    test_preds.append(pred)

test_preds = pnp.array(test_preds)
y_test_transformed = y_test_orig * 2 - 1  # {0,1} -> {-1, +1}

acc_test = accuracy_score(y_test_transformed, test_preds)
f1_test = f1_score(y_test_transformed, test_preds)

print("=== Performance du VQC sur le test set ===")
print(f"Accuracy : {acc_test:.4f}")
print(f"F1-score : {f1_test:.4f}")

cm = confusion_matrix(y_test_transformed, test_preds)
print("\nMatrice de confusion :")
print(cm)

---
## Partie C : VQC avec PennyLane + PyTorch (qml.qnn.TorchLayer)

On utilise `qml.qnn.TorchLayer` pour intégrer le circuit comme une couche PyTorch.

In [ ]:
# VQC comme couche PyTorch via TorchLayer

n_qubits_torch = 4
n_layers_torch = 3

dev_torch = qml.device('default.qubit', wires=n_qubits_torch)

@qml.qnode(dev_torch)
def circuit_torch(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits_torch), rotation='Y')
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits_torch))
    return qml.expval(qml.PauliZ(0))

weight_shapes = {'weights': (n_layers_torch, n_qubits_torch, 3)}

qlayer = qml.qnn.TorchLayer(circuit_torch, weight_shapes)

# Modèle complet
class VQC_torch(nn.Module):
    def __init__(self):
        super().__init__()
        self.qlayer = qml.qnn.TorchLayer(circuit_torch, weight_shapes)

    def forward(self, x):
        return self.qlayer(x)

vqc_model = VQC_torch()
print("Modèle VQC PyTorch créé avec succès.")
print(vqc_model)

# Test
x_torch_test = torch.tensor([[0.5, 1.2, 0.8, 2.0]], dtype=torch.float32)
out = vqc_model(x_torch_test)
print(f"\nSortie du modèle : {out.item():.4f}")

In [ ]:
# Entraînement du VQC avec PyTorch

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

vqc_model_torch = VQC_torch()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(vqc_model_torch.parameters(), lr=0.1)

n_epochs_torch = 80
loss_torch = []
acc_torch = []

print("Entraînement VQC (TorchLayer) ...")
for epoch in range(n_epochs_torch):
    vqc_model_torch.train()
    optimizer.zero_grad()
    outputs = vqc_model_torch(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    
    loss_torch.append(loss.item())
    
    with torch.no_grad():
        preds = (outputs >= 0).float()
        acc = (preds == y_train_t).float().mean().item()
        acc_torch.append(acc)
    
    if (epoch + 1) % 10 == 0:
        print(f"Époque {epoch+1:3d}/{n_epochs_torch} — Loss : {loss.item():.4f} — Acc : {acc:.4f}")

# Évaluation test
vqc_model_torch.eval()
with torch.no_grad():
    test_outputs = vqc_model_torch(X_test_t)
    test_preds = (test_outputs >= 0).float()
    acc_test_torch = (test_preds == y_test_t).float().mean().item()
    print(f"\nTest accuracy (TorchLayer) : {acc_test_torch:.4f}")

---
## Partie D : Comparaison avec la baseline classique

On compare le VQC avec les modèles classiques du Lab 2 sur les mêmes données.

In [ ]:
# Classifieurs classiques sur les mêmes données

svm_classique = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_classique.fit(X_train_scaled, y_train)
y_pred_svm = svm_classique.predict(X_test_scaled)

acc_svm = accuracy_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)

print("=== Comparaison VQC vs. Classique ===")
print(f"{'Modèle':<25} {'Accuracy':<12} {'F1-score':<12}")
print('-' * 49)
print(f"{'SVM RBF (classique)':<25} {acc_svm:<12.4f} {f1_svm:<12.4f}")
print(f"{'VQC PennyLane':<25} {acc_test:<12.4f} {f1_test:<12.4f}")
print(f"{'VQC TorchLayer':<25} {acc_test_torch:<12.4f} {'N/A':<12}")

print()
print("Note : Le VQC atteint des performances comparables au SVM classique")
print("sur ce dataset simple. L'intérêt du VQC réside dans l'expressivité")
print("des feature maps quantiques pour des données plus complexes.")

In [ ]:
# Visualisation comparative

modeles_comp = {
    'SVM RBF': ('#1f77b4', acc_svm),
    'VQC (PennyLane)': ('#ff7f0e', acc_test),
    'VQC (TorchLayer)': ('#2ca02c', acc_test_torch),
}

fig, ax = plt.subplots(figsize=(8, 4))
noms = list(modeles_comp.keys())
couleurs = [modeles_comp[n][0] for n in noms]
accs = [modeles_comp[n][1] for n in noms]

bars = ax.bar(noms, accs, color=couleurs, alpha=0.8, edgecolor='k')
ax.set_ylabel('Test Accuracy')
ax.set_title('Comparaison VQC vs. Classique — Iris (2 classes)')
ax.set_ylim(0, 1.05)
ax.axhline(1.0, color='gray', ls='--', alpha=0.5, label='Perfect')

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{acc:.3f}', ha='center', va='bottom')

ax.legend()
plt.tight_layout()
plt.show()

---
## Partie E : Exercices

### Exercice 1 : Tester différents ansätze

Remplacez `BasicEntanglerLayers` par :
- `StronglyEntanglingLayers`
- Un circuit personnalisé avec des portes $R_X$-$R_Y$-$R_Z$ alternées

Comparez les performances.

In [ ]:
# Exercice 1 : Ansätze alternatifs

def creer_vqc_perso(use_strongly=False):
    """Crée un VQC avec un ansatz différent."""
    dev_ex = qml.device('default.qubit', wires=n_qubits)
    
    @qml.qnode(dev_ex)
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation='Y')
        if use_strongly:
            qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        else:
            # Ansatz personnalisé : couches de rotations alternées
            for layer in range(weights.shape[0]):
                for q in range(n_qubits):
                    qml.RX(weights[layer, q, 0], wires=q)
                    qml.RY(weights[layer, q, 1], wires=q)
                    qml.RZ(weights[layer, q, 2], wires=q)
                for q in range(n_qubits - 1):
                    qml.CNOT(wires=[q, q + 1])
        return qml.expval(qml.PauliZ(0))
    
    return circuit

print("Circuit StronglyEntanglingLayers :")
qml.draw_mpl(creer_vqc_perso(use_strongly=True))(test_inputs, init_weights)
plt.show()

print("Circuit personnalisé :")
qml.draw_mpl(creer_vqc_perso(use_strongly=False))(test_inputs, init_weights)
plt.show()

### Exercice 2 : Effet de la régularisation et du learning rate

Testez l'impact du learning rate (0.01, 0.05, 0.1, 0.5) sur la convergence du VQC.

In [ ]:
# Exercice 2 : Impact du learning rate

learning_rates = [0.01, 0.05, 0.1, 0.5]
losses_lr = {}

for lr in learning_rates:
    w = pnp.random.uniform(-np.pi, np.pi, (n_layers, n_qubits, 3), requires_grad=True)
    opt_lr = qml.AdamOptimizer(stepsize=lr)
    losses = []
    for _ in range(50):
        w, loss = opt_lr.step_and_cost(
            lambda w: cross_entropy_loss(w, X_train_pl, y_train_pl), w
        )
        losses.append(loss)
    losses_lr[lr] = losses
    print(f"LR={lr:.2f} — Loss finale : {losses[-1]:.4f}")

# Visualisation
plt.figure(figsize=(8, 5))
for lr, losses in losses_lr.items():
    plt.plot(losses, label=f'LR={lr}')
plt.xlabel('Époque')
plt.ylabel('Loss')
plt.title('Impact du learning rate sur la convergence')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Exercice 3 : VQC pour classification multi-classe (3 classes Iris)

Utilisez 3 qubits de sortie (un par classe) avec un softmax pour classer les 3 classes d'Iris.

In [ ]:
# Exercice 3 : VQC multi-classe

n_qubits_mc = 4
dev_mc = qml.device('default.qubit', wires=n_qubits_mc + 1)  # +1 qubit de mesure

@qml.qnode(dev_mc)
def circuit_multiclass(inputs, params):
    qml.AngleEmbedding(inputs, wires=range(n_qubits_mc))
    
    # 3 couches de StronglyEntanglingLayers
    qml.StronglyEntanglingLayers(params['W1'], wires=range(n_qubits_mc))
    
    # Mesure sur 3 qubits différents
    return [qml.expval(qml.PauliZ(i)) for i in range(3)]

print("Architecture multi-classe (3 sorties) :")
print("  Chaque sortie correspond à une classe.")
print("  Prédiction = argmax des 3 espérances.")

### Exercice 4 : Nombre de couches et barren plateaux

Augmentez le nombre de couches (1, 2, 5, 10) et observez l'impact sur la convergence.

In [ ]:
# Exercice 4 : Impact du nombre de couches

for n_layers_test in [1, 3, 5, 8]:
    w = pnp.random.uniform(-np.pi, np.pi, (n_layers_test, n_qubits, 3), requires_grad=True)
    
    @qml.qnode(dev)
    def circuit_test(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits))
        qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
        return qml.expval(qml.PauliZ(0))
    
    opt_test = qml.AdamOptimizer(stepsize=0.1)
    for epoch in range(30):
        w, _ = opt_test.step_and_cost(
            lambda w: cross_entropy_loss(w, X_train_pl, y_train_pl), w
        )
    
    final_loss = cross_entropy_loss(w, X_train_pl, y_train_pl)
    print(f"Couches={n_layers_test:2d} — Loss finale : {final_loss:.4f}")

---
## Pour aller plus loin

- Implémentez un VQC avec `qml.qnn.KerasLayer` pour TensorFlow
- Essayez l'optimiseur SPSA (Simultaneous Perturbation Stochastic Approximation)
- Ajoutez un `qml.RandomLayers` comme ansatz
- Comparez la performance du VQC avec et sans entangling layers
- Références : [SP21] Ch. 5 — *Machine Learning with Quantum Computers*, [VQA26] — *A Review of Variational Quantum Algorithms*